In [7]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("data/raw")
PROC = Path("data/processed")
PROC.mkdir(parents=True, exist_ok=True)

# helper to load CSV safely
def load_csv(name):
    return pd.read_csv(RAW / name)

In [3]:
# helper to load CSV safely
def load_csv(name):
    return pd.read_csv(RAW / name)

In [8]:
import os
os.getcwd()

'c:\\F1-Project'

In [10]:
results = load_csv("results.csv")
# Inspect
results.head()
# Ensure columns: raceId, driverId, position (string) or positionOrder (numeric)
# Convert position to numeric (some datasets have '1','2','NC' etc.)
results['position'] = pd.to_numeric(results['position'], errors='coerce')

# If you want bucketed label: podium / top10 / outside
def finish_bucket(pos):
    if pd.isna(pos): return "DNF"
    pos = int(pos)
    if pos <= 3: return "podium"
    if pos <= 10: return "top10"
    return "outside"

results['finish_bucket'] = results['position'].apply(finish_bucket)

# Save a race-driver table (one row per race-driver) with labels
race_driver = results[['raceId','driverId','constructorId','grid','position','finish_bucket','points']]
race_driver.to_csv(PROC / "race_driver_labels.csv", index=False)

In [11]:
laps = load_csv("lap_times.csv")
# Typical columns: raceId, driverId, lap, position, time, milliseconds
laps = laps.rename(columns={'time':'lap_time_str','milliseconds':'lap_time_ms'})
laps['lap_time_s'] = laps['lap_time_ms'] / 1000.0  # seconds

# Example filtering: remove laps with NaN milliseconds or pit-in/out rows
laps = laps.dropna(subset=['lap_time_ms'])

# Save processed lap-level table
laps.to_csv(PROC / "lap_times_processed.csv", index=False)


In [13]:
races = load_csv("races.csv")
# assume column 'safetyCar' exists with 0/1 or 'Yes'...
if 'safetyCar' in races.columns:
    races['safety_car'] = races['safetyCar'].astype(int)
else:
    # Alternative heuristic: check results for any 'Safety Car' in notes (rare)
    # For now add placeholder 0 and you can later fill by manual lookup or another dataset
    races['safety_car'] = 0

races[['raceId','year','round','circuitId','date','safety_car']].to_csv(PROC / "race_safetycar_labels.csv", index=False)

In [16]:
# results already loaded above
# ensure results has constructorId and points numeric
results['points'] = pd.to_numeric(results['points'], errors='coerce').fillna(0.0)
constructor_points_race = results.groupby(['raceId','constructorId'], as_index=False)['points'].sum()
constructor_points_race.rename(columns={'points':'constructor_points_race'}, inplace=True)
constructor_points_race.to_csv(PROC / "constructor_points_per_race.csv", index=False)

races = load_csv("races.csv")
# join race->season onto constructor_points_race
constructor_points = constructor_points_race.merge(races[['raceId','year']], on='raceId', how='left')
constructor_points_season = constructor_points.groupby(['year','constructorId'], as_index=False)['constructor_points_race'].sum()
constructor_points_season.rename(columns={'constructor_points_race':'constructor_points_season'}, inplace=True)
constructor_points_season.to_csv(PROC / "constructor_points_season.csv", index=False)
